# Prediction
This notebook will be used to prepare, train, and evaluate a simple classification model to predict the administrations of single or multiple doses on a patient using age, gender, ward (location), hour, day of week, month, and year.

Accordingly to the analysis, the model shouldn't be able to make very certain predictions based on those features, as the severity (administrations of multiple doses) of an incident doesn't have a explicit, well-defined pattern. It in fact exists, but is definitely not strong. So what I'm expecting is a model that is not random (50/50), but also not good.

The plan is to use a Classification Decision Tree to predict a binary (boolean) output -> is multiple dose or not.

In [1]:
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("../data/naloxone_clean.csv")

## Selecting Features

In [2]:
selected_columns = [
    "year",
    "month",
    "day_of_week",
    "hour",
    "age",
    "gender",
    "ward",
    "is_multiple_dose"
]

df = df[selected_columns]
df

,year,month,day_of_week,hour,age,gender,ward,is_multiple_dose
0,2021,8,6,4,30 to 34,Male,NaN,False
1,2021,8,6,7,55 to 59,Female,Fort Rouge - East Fort Garry,False
2,2021,8,6,16,50 to 54,Female,Fort Rouge - East Fort Garry,False
3,2021,8,6,20,35 to 39,Female,NaN,True
4,2021,8,6,21,25 to 29,Male,Point Douglas,False
...,...,...,...,...,...,...,...,...
26317,2026,3,2,15,25 to 29,Male,Point Douglas,False
26318,2026,3,2,16,30 to 34,Male,Point Douglas,True
26319,2026,3,3,0,55 to 59,Male,Daniel McIntyre,True
26320,2026,3,3,18,30 to 34,Female,Mynarski,False


## Dealing with missing values

Decision Trees doesn't require a standardized data because they don't make decisions based on distance, so one less thing to worry about. However, they don't know how to deal with missing values (NaN) naturally, and I have a bunch of them on the dataset on the `age`, `gender`, and `ward` columns. So what I'll do is bring back the `"Unknown"` values for replacing those missing values. That way I don't loose the "no location recorded" information that can be useful.

In [3]:
df.isna().sum()

year                  0
month                 0
day_of_week           0
hour                  0
age                 284
gender              203
ward                338
is_multiple_dose      0
dtype: int64

In [4]:
df = df.fillna("Unknown")

In [5]:
df.isna().sum()

year                0
month               0
day_of_week         0
hour                0
age                 0
gender              0
ward                0
is_multiple_dose    0
dtype: int64

## Splittinng into features / target

In [6]:
X = df.drop(columns=["is_multiple_dose"]) # features
y = df["is_multiple_dose"] # target

## Splitting into train / test (80-20)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("X_train rows:", len(X_train))
print("X_test rows:", len(X_test))

X_train rows: 21057
X_test rows: 5265


## Encoding the categorical features
Some features are categorical: `age` (string age range), and `gender`. They have to be numbers so the decision tree can interpret them. I'll be using OneHotEncoder to encode these features. It creates a new column for each unique category, being binary, 1 for True and 0 for False.

In [8]:
cat_cols = ["age", "gender", "ward"]

oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
oh.fit(X_train[cat_cols])

train_enc = oh.transform(X_train[cat_cols])
test_enc  = oh.transform(X_test[cat_cols])

feature_names = oh.get_feature_names_out(cat_cols)

train_enc = pd.DataFrame(train_enc, columns=feature_names, index=X_train.index)
test_enc  = pd.DataFrame(test_enc, columns=feature_names, index=X_test.index)

num_cols = ["year", "month", "day_of_week", "hour"]

X_train_final = pd.concat([X_train[num_cols], train_enc], axis=1)
X_test_final  = pd.concat([X_test[num_cols], test_enc], axis=1)

## Training the model
As I known from exploring the data that the split of single/multiple dose is not proportional, I'll have to tune `class_weight` hyperparameter so the model gives more importance to the multiple dose class. This will bring more false positives, but is expected and normal.

I'll also tune the "mandatory" hyperparameter for Decision Trees, `max_depth`. This parameter prevent the tree from growing for ever and memorizing the training data and its noise -> overfitting.

In [9]:
dt = DecisionTreeClassifier(
    class_weight="balanced",
    max_depth=10,
    random_state=42
)

dt.fit(X_train_final, y_train)
y_pred = dt.predict(X_test_final)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))

Accuracy: 0.6009
F1: 0.4798
Recall: 0.5327
Precision: 0.4365
